In [ ]:
# --- repo bootstrap: makes this notebook runnable from anywhere ---
import os, sys
ROOT = os.getcwd()
while ROOT != os.path.dirname(ROOT) and not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = "/kaggle/input/datasets/sinadaneshgar/edgedeid"  # Kaggle dataset fallback
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT)

In [3]:
import os
import pandas as pd
import re
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import datetime
from xml.dom import minidom
from typing import Dict, List, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import sys
from edgedeid.data.preprocessing import preprocess_text
from edgedeid.data.xml_utils import convert_to_xml
from edgedeid.prompts import SYSTEM_PROMPT

EVAL_SYSTEM = "full" # full or quantized

In [4]:
eval_df = pd.read_csv("evals/ext_eval/en_data.csv")
eval_df.head(2)

,Report,DeID
0,REPORT\nXR CHEST Exam: XR Chest 2 Views PA + ...,REPORT\nXR CHEST Exam: XR Chest 2 Views PA + ...
1,REPORT\nXR CHEST Exam: XR Chest 1 View Portab...,REPORT\nXR CHEST Exam: XR Chest 1 View Portab...


## Preprocessing

In [5]:
eval_df = preprocess_text(eval_df, input_col="Report", output_col="DeID")

Starting text processing pipeline...
Fixing spaces in closing tags...
Fixing spaces in tags...
Processing phone numbers...
Processing dates...
Processing ages...
Correcting gender terms...
Processing name tags...
Simplifying name tags...
Reformatting ID tags...
Processing remaining dates...
Removing markdown tags...

Processing Summary:
Phone numbers formatted: 0
Dates formatted: 0 + 0
Ages processed: 0
Gender terms corrected: 0
Name tags processed: 16
Name types simplified: 4
ID tags reformatted: 0
Markdown tags removed: 0

Text processing pipeline completed!


In [6]:
eval_df['GT-XML'] = eval_df["DeID"].apply(convert_to_xml)
eval_df["Report"] = eval_df["Report"].str.replace("<anonymized_report>","").str.replace("</anonymized_report>","").str.strip().str.replace("<anonymized_report/>", "").str.replace("<raw_report>","").str.replace("</raw_report>","").str.strip().str.replace("<raw_report/>", "")
eval_df.head(2)

,Report,DeID,GT-XML
0,REPORT\nXR CHEST Exam: XR Chest 2 Views PA + ...,REPORT\nXR CHEST Exam: XR Chest 2 Views PA + ...,<phi_entities>\n<names/>\n<ages>\n<age>54</age...
1,REPORT\nXR CHEST Exam: XR Chest 1 View Portab...,REPORT\nXR CHEST Exam: XR Chest 1 View Portab...,<phi_entities>\n<names/>\n<ages>\n<age>59</age...


Run:
```bash
VLLM_USE_V1="0" CUDA_VISIBLE_DEVICES="3" python -m vllm.entrypoints.openai.api_server --model ../../SFT/Qwen3-0.6B-Base_SFT --port 9995 --tensor-parallel-size 1 --max-model-len 4096 --generation-config vllm --dtype "bfloat16"
```
or
```bash
VLLM_USE_V1="0" CUDA_VISIBLE_DEVICES="3" python -m vllm.entrypoints.openai.api_server --model ../../Final_model/Qwen3-0.6B-Base_SFT-Q4_K_M.gguf --port 9995 --tokenizer "../../SFT/Qwen3-0.6B-Base_SFT"  --tensor-parallel-size 1 --max-model-len 4096 --generation-config vllm
```

In [7]:
from openai import OpenAI
client = OpenAI(api_key="EMPTY", base_url="http://localhost:9995/v1")
# generation task
def run_model(note, examples=[], temperature=0):
    global client

    # Build examples string if examples exist, following sft.ipynb style
    examples_str = ""
    if examples and examples != "NONE":
        examples_str_list = [
            f"""<example_note>
{example['original']}
</example_note>

<example_phi_entities>
{example['anonymized']}
</example_phi_entities>""" 
            for example in examples
        ]
        examples_str = "\n\n".join(examples_str_list)
        examples_str = f"<examples>\n{examples_str}\n</examples>\n\n"

    input_text = \
f"""<instructions>
{SYSTEM_PROMPT}
</instructions>

{examples_str}<note>
{note.strip()}
</note>

<phi_entities>
"""

    response = client.completions.create(
        model="SFT/Qwen3-0.6B-Base_SFT" if EVAL_SYSTEM == "full" else "Final_model/Qwen3-0.6B-Base_SFT-Q4_K_M.gguf",
        prompt=input_text,
        max_tokens=4096,
        temperature=temperature,
        seed=2025,
        stop=["</phi_entities>"]
    )
    
    # return response.choices[0].text
    response_text = "<phi_entities>\n" + response.choices[0].text
    response_text = response_text.split("</phi_entities>", 1)[0] + "</phi_entities>"
    response_text = response_text

    return response_text

In [8]:
eval_df[f"LLM-XML"] = ""

def process_row(i):
    return i, run_model(eval_df.at[i, "Report"], temperature=0.0)

with ThreadPoolExecutor(max_workers=100 if EVAL_SYSTEM == "full" else 1) as executor:
    futures = [executor.submit(process_row, i) for i in range(len(eval_df))]
    for future in tqdm(as_completed(futures), total=len(eval_df), desc="Processing rows"):
        i, result = future.result()
        eval_df.at[i, f"LLM-XML"] = result

Processing rows:   0%|          | 0/100 [00:01<?, ?it/s]


APIConnectionError: Connection error.

In [ ]:
eval_df

In [ ]:
sample = """REPORT
XR Chest 1 View Portable
CLINICAL INDICATION:  Cough. sob
TECHNIQUE: Structured report code: 17.ER2
COMPARISON: Barium swallow dated June 26, 2014.
FINDINGS: Support Devices:  None.
Heart/Mediastinum:  Retrocardiac air soft tissue density potentially within the esophagus. Cardiac and mediastinal contours are otherwise within normal limits.
Lungs/Pleura:  No pulmonary edema. No airspace opacity. No pleural effusion. No pneumothorax.
Bones/Soft Tissues:  No acute abnormality.
IMPRESSION: Clear lungs. Retrocardiac opacity on the frontal projection corresponds to known moderate size hiatal hernia and air within the esophagus."""

print(run_model(sample))

In [ ]:
def extract_entities(xml_str):
    entities = defaultdict(list)
    try:
        # Check if input is a valid string
        if not isinstance(xml_str, str):
            return entities
        
        parent_tags = ['names', 'ages', 'dates', 'places', 'addresses', 'phones', 'emails', 'ids', 'urls', 'devices']
        root = ET.fromstring(xml_str)
        for elem in root:
            if elem.tag != root.tag:  # Skip the root tag
                if elem.tag in parent_tags:
                    for child in elem:
                        if child.tag == "name":
                            variations = child.find("variations")
                            if variations is not None:
                                values = [value.text.strip() for value in variations.findall("value") if value.text]
                                entities[child.tag].extend(values)
                        elif child.tag == "date":
                            try:
                                # Assuming the date is in a standard format, e.g., YYYY-MM-DD
                                formatted_date = datetime.strptime(child.find("formatted").text, "%Y-%m-%d").strftime("%Y-%m-%d")
                                entities[child.tag].append(formatted_date)
                            except (ValueError, AttributeError):
                                pass
                        else:
                            entities[child.tag].append(child.text.strip() if child.text else "")
    except ET.ParseError:
        pass
    return entities

def compare_entities(gt_xml, llm_xml):
    # Handle cases where either input is not a string
    if not isinstance(gt_xml, str) or not isinstance(llm_xml, str):
        return {}, {}
        
    try:
        llm_xml = re.search(r'(<phi_entities>[\s\S]*?</phi_entities>)', llm_xml).group(1)
        llm_xml = llm_xml.split('</phi_entities>')[0] + '</phi_entities>'
    except (AttributeError, TypeError):
        return {}, {}

    gt_entities = extract_entities(gt_xml)
    llm_entities = extract_entities(llm_xml)
        
    gt_tag_stats = {}
    for tag, gt_values in gt_entities.items():
        gt_tag_stats[tag] = {"correct": 0, "total": 0}
        llm_values = llm_entities.get(tag, [])
        for value in gt_values:
            gt_tag_stats[tag]['total'] += 1
            if value.lower() in [llm_value.lower() for llm_value in llm_values]:
                gt_tag_stats[tag]['correct'] += 1
            else:
                for llm_value in llm_values:
                    if value.lower() in llm_value.lower():
                        gt_tag_stats[tag]['correct'] += 1
                        break
                else:
                    print("*"*100)
                    print("False Negative:")
                    print("GT:", value)
                    print("LLM:", llm_values)
                    print("*"*100)
    
    predicted_tag_stats = {}
    for tag, llm_values in llm_entities.items():
        predicted_tag_stats[tag] = {"correct": 0, "total": 0}
        gt_values = gt_entities.get(tag, [])
        for value in llm_values:
            predicted_tag_stats[tag]['total'] += 1
            if any(value.lower() in gt_value.lower() or gt_value.lower() in value.lower() for gt_value in gt_values):
                predicted_tag_stats[tag]['correct'] += 1
            else:
                print("*"*100)
                print("False Positive:")
                print("GT:", gt_values)
                print("LLM:", llm_values)
                print("*"*100)
    
    return gt_tag_stats, predicted_tag_stats

def calculate_tag_metrics(gt_tag_stats, predicted_tag_stats):
    """
    Calculate precision, recall, and F1 score for each tag type.
    """
    tag_metrics = {}
    all_tags = set(list(gt_tag_stats.keys()) + list(predicted_tag_stats.keys()))
    
    for tag in all_tags:
        gt_stats = gt_tag_stats.get(tag, {"correct": 0, "total": 0})
        pred_stats = predicted_tag_stats.get(tag, {"correct": 0, "total": 0})
        
        gt_correct = gt_stats['correct']
        gt_total = gt_stats['total']
        pred_correct = pred_stats['correct']
        pred_total = pred_stats['total']
        
        # Calculate recall
        recall = gt_correct / gt_total if gt_total > 0 else 0
        
        # Calculate precision
        precision = pred_correct / pred_total if pred_total > 0 else 0
        
        # Calculate F1 score
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        tag_metrics[tag] = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'gt_total': gt_total,
            'pred_total': pred_total,
            'correct': gt_correct
        }
    
    return tag_metrics

def subgroup_analysis(eval_df):
    """
    Perform detailed subgroup analysis including tag-level metrics.
    """
    accumulated_gt_stats = {}
    accumulated_pred_stats = {}
    
    for i, row in eval_df.iterrows():
        gt_xml = str(row['GT-XML'])
        llm_xml = str(row['LLM-XML'])
        
        if not llm_xml.startswith("<phi_entities>"):
            continue
        
        print(i)
        gt_tag_stats, predicted_tag_stats = compare_entities(gt_xml, llm_xml)
        
        # Accumulate statistics for each tag
        for tag in set(list(gt_tag_stats.keys()) + list(predicted_tag_stats.keys())):
            if tag not in accumulated_gt_stats:
                accumulated_gt_stats[tag] = {"correct": 0, "total": 0}
            if tag not in accumulated_pred_stats:
                accumulated_pred_stats[tag] = {"correct": 0, "total": 0}
                
            if tag in gt_tag_stats:
                accumulated_gt_stats[tag]['correct'] += gt_tag_stats[tag]['correct']
                accumulated_gt_stats[tag]['total'] += gt_tag_stats[tag]['total']
            if tag in predicted_tag_stats:
                accumulated_pred_stats[tag]['correct'] += predicted_tag_stats[tag]['correct']
                accumulated_pred_stats[tag]['total'] += predicted_tag_stats[tag]['total']
    
    # Calculate overall metrics
    total_gt_correct = sum(stats['correct'] for stats in accumulated_gt_stats.values())
    total_gt = sum(stats['total'] for stats in accumulated_gt_stats.values())
    total_pred_correct = sum(stats['correct'] for stats in accumulated_pred_stats.values())
    total_pred = sum(stats['total'] for stats in accumulated_pred_stats.values())
    
    overall_recall = total_gt_correct / total_gt if total_gt > 0 else 0
    overall_precision = total_pred_correct / total_pred if total_pred > 0 else 0
    overall_f1 = 2 * (overall_precision * overall_recall) / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0
    
    # Calculate tag-level metrics
    tag_metrics = calculate_tag_metrics(accumulated_gt_stats, accumulated_pred_stats)
    
    overall_metrics = {
        'precision': overall_precision,
        'recall': overall_recall,
        'f1': overall_f1,
        'total_correct': total_gt_correct,
        'total_gt': total_gt,
        'total_pred': total_pred
    }
    
    return overall_metrics, tag_metrics

def print_metrics_report(overall_metrics, tag_metrics):
    """
    Print a formatted report of all metrics.
    """
    print("="*50)
    print("OVERALL METRICS")
    print("="*50)
    print(f"Precision: {overall_metrics['precision']:.4f}")
    print(f"Recall: {overall_metrics['recall']:.4f}")
    print(f"F1 Score: {overall_metrics['f1']:.4f}")
    print(f"Total Correct: {overall_metrics['total_correct']}")
    print(f"Total Ground Truth: {overall_metrics['total_gt']}")
    print(f"Total Predicted: {overall_metrics['total_pred']}")
    
    print("\n" + "="*50)
    print("TAG-LEVEL METRICS")
    print("="*50)
    
    # Calculate maximum tag length for formatting
    max_tag_len = max(len(str(tag)) for tag in tag_metrics.keys())
    
    # Print header
    header = f"{'Tag':<{max_tag_len}} | {'Precision':>10} | {'Recall':>10} | {'F1':>10} | {'GT Count':>10} | {'Pred Count':>10}"
    print(header)
    print("-" * len(header))
    
    # Print metrics for each tag
    for tag, metrics in sorted(tag_metrics.items()):
        print(f"{str(tag):<{max_tag_len}} | {metrics['precision']:>10.4f} | {metrics['recall']:>10.4f} | "
              f"{metrics['f1']:>10.4f} | {metrics['gt_total']:>10d} | {metrics['pred_total']:>10d}")

overall_metrics, tag_metrics = subgroup_analysis(eval_df)
print_metrics_report(overall_metrics, tag_metrics)

In [ ]:
print(eval_df.iloc[8]['Report'])

In [ ]:
import os
os.makedirs("results", exist_ok=True)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Use the tag_metrics from calculate_tag_metrics output
metrics = tag_metrics

# Calculate overall accuracy using the new structure
total_correct = sum(tag['correct'] for tag in metrics.values())
total_gt = sum(tag['gt_total'] for tag in metrics.values())
overall_accuracy = total_correct / total_gt if total_gt > 0 else 0

print(f"Overall Accuracy: {overall_accuracy:.4f}")

# Print tag-level accuracy
print("\nTag-level Accuracy:")
for tag, values in metrics.items():
    tag_accuracy = values['recall']  # Recall is equivalent to accuracy in this context
    print(f"{tag}: {tag_accuracy:.4f}")

# Prepare data for bar plot
tags = list(metrics.keys())
correct_counts = [values['correct'] for values in metrics.values()]
total_counts = [values['gt_total'] for values in metrics.values()]
incorrect_counts = [total - correct for total, correct in zip(total_counts, correct_counts)]

# Prepare data for spider plot - use recall values
tag_recalls = [values['recall'] for values in metrics.values()]

# Create a subplot with 1 row and 2 columns, specifying the type for each subplot
fig = make_subplots(
    rows=1, cols=2, 
    specs=[[{"type": "xy"}, {"type": "polar"}]]  # Specify subplot types
)

# Add bar plot to the first column
fig.add_trace(go.Bar(name='Total', x=tags, y=total_counts, marker_color='blue'), row=1, col=1)
fig.add_trace(go.Bar(name='Correct', x=tags, y=correct_counts, marker_color='green'), row=1, col=1)
fig.add_trace(go.Bar(name='Incorrect', x=tags, y=incorrect_counts, marker_color='red'), row=1, col=1)

# Add spider plot to the second column
fig.add_trace(go.Scatterpolar(r=tag_recalls, theta=tags, fill='toself', name='Recall'), row=1, col=2)

# Update layout
fig.update_layout(
    barmode='group',
    title='PHI Tag Performance and Recall',
    font=dict(size=16),
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0.7, 1]  # Scale from 70% to 100%
        )
    ),
    showlegend=True,
    legend=dict(
        x=1, y=1,  # Position the legend at the top right
        yanchor='top'
    ),
    height=600,  # Set the height of the chart
    width=1200   # Set the width of the chart
)

# Show the composite plot
fig.show()

# Save the chart as an HTML file
fig.write_html("results/phi_tag_performance_composite.html")
print("Composite chart saved as 'phi_tag_performance_composite.html'")